In [0]:
import dlt
from pyspark.sql.functions import *

# -------------------------------------------------------------------
# 1. TIME & BUSINESS LEVEL METRICS
# -------------------------------------------------------------------

@dlt.table(
    name="hourly_order_metrics",
    comment="Hourly aggregated metrics for ML anomaly detection and operational monitoring"
)
def hourly_order_metrics():
    return (
        dlt.read("clean_orders") # Reads from Silver
        .groupBy(window("event_timestamp", "1 hour"))
        .agg(
            count("order_id").alias("total_orders"),
            sum(expr("quantity * unit_price")).alias("total_revenue"),
            avg("discount").alias("avg_discount")
        )
        .select(
            col("window.start").alias("hour_start"),
            "total_orders",
            "total_revenue",
            "avg_discount"
        )
    )

@dlt.table(
    name="daily_business_metrics",
    comment="High-level daily rollups for general business health (Revenue, Orders, Data Quality)"
)
def daily_business_metrics():
    return (
        dlt.read("clean_orders")
        .groupBy("event_date")
        .agg(
            sum(expr("quantity * unit_price")).alias("total_daily_revenue"),
            countDistinct("order_id").alias("total_daily_orders"),
            sum("quantity").alias("total_units_sold"),
            avg("quality_score").alias("avg_quality_score"), 
            avg("discount").alias("avg_discount_given"),

        )
    )

# -------------------------------------------------------------------
# 2. CATEGORY LEVEL METRICS
# -------------------------------------------------------------------

@dlt.table(
    name="category_summary",
    comment="Lifetime summary of product categories"
)
def category_summary():
    return (
        dlt.read("clean_orders")
        .groupBy("category")
        .agg(
            count("order_id").alias("total_category_orders"),
            sum(expr("quantity * unit_price")).alias("total_category_revenue")
        )
    )

@dlt.table(
    name="category_daily_metrics",
    comment="Daily aggregations by category for 'Top category yesterday/today' questions"
)
def category_daily_metrics():
    return (
        dlt.read("clean_orders")
        .groupBy("event_date", "category")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            sum(expr("quantity * unit_price")).alias("total_revenue"),
            sum("quantity").alias("total_units_sold")
        )
    )

# -------------------------------------------------------------------
# 3. PRODUCT LEVEL METRICS
# -------------------------------------------------------------------

@dlt.table(
    name="product_performance",
    comment="Lifetime performance of individual products"
)
def product_performance():
    return (
        dlt.read("clean_orders")
        .groupBy("product_id", "category")
        .agg(
            sum("quantity").alias("total_units_sold"),
            sum(expr("quantity * unit_price")).alias("total_product_revenue"),
            avg("discount").alias("avg_discount_given"),
            countDistinct("order_id").alias("times_ordered")
        )
    )

@dlt.table(
    name="product_daily_metrics",
    comment="Daily aggregations by product for 'Top products today/yesterday' questions"
)
def product_daily_metrics():
    return (
        dlt.read("clean_orders")
        .groupBy("event_date", "product_id", "category")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            sum(expr("quantity * unit_price")).alias("total_revenue"),
            sum("quantity").alias("total_units_sold"), 
            avg("discount").alias("avg_discount_given"),

        )
    )

# -------------------------------------------------------------------
# 4. CUSTOMER LEVEL METRICS (CUSTOMER 360)
# -------------------------------------------------------------------

@dlt.table(
    name="customer_360",
    comment="Customer-level aggregations including lifetime spend, preferences, and order history"
)
def customer_360():
    return (
        dlt.read("clean_orders")
        .groupBy("customer_id")
        .agg(
            sum(expr("(quantity * unit_price) - discount")).alias("total_lifetime_spend"),
            countDistinct("order_id").alias("total_orders_placed"),
            min("event_date").alias("first_order_date"),
            max("event_date").alias("latest_order_date"),
            
            # Using mode() to find the most frequent value for the customer
            expr("mode(device_type)").alias("preferred_device"),
            expr("mode(payment_method)").alias("preferred_payment_method"),
            expr("mode(category)").alias("preferred_category")
        )
        # Calculate Average Order Value (AOV)
        .withColumn("avg_order_value", round(col("total_lifetime_spend") / col("total_orders_placed"), 2))
    )